# Kiểm thử API RAGProvider (Graph Database Legal Assistant)

- **Thư mục root**: `ML_final`
- **Mô hình Graph**: Local Graph Database (`src/db/graph_database`) hoặc Neo4j Instance
- **Hỗ trợ 2 chế độ**: `rag=True` (w/ RAG) và `rag=False` (w/o RAG - Direct LLM)

Notebook này kiểm thử trực tiếp các endpoints của **FastAPI RAGProvider** qua `TestClient`.

In [ ]:
import os
import sys
import json
from pathlib import Path

# Đảm bảo ML_final nằm trong sys.path và là working directory
CURRENT_DIR = Path.cwd()
for p in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if p.name == "ML_final" or (p / "src" / "be").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        os.chdir(str(p))
        break

print(f"📁 Thư mục làm việc hiện tại: {Path.cwd()}")

from fastapi.testclient import TestClient
from src.be.RAGProvider.api import app

client = TestClient(app)
print("✅ Đã khởi tạo TestClient RAGProvider thành công!")

## 1. Kiểm tra trạng thái hệ thống (`GET /health`)

In [ ]:
res_health = client.get("/health")
print(f"Status Code: {res_health.status_code}")
print(json.dumps(res_health.json(), indent=2, ensure_ascii=False))

## 2. Kiểm thử chế độ w/o RAG (`rag=False` -> Direct LLM Answer)

In [ ]:
direct_payload = {
    "query": "Quyền sử dụng đất là gì?",
    "rag": False,
    "session_id": "test_direct_001"
}

res_direct = client.post("/query", json=direct_payload)
print(f"Status Code: {res_direct.status_code}")
print(json.dumps(res_direct.json(), indent=2, ensure_ascii=False))

## 3. Kiểm thử chế độ w/ RAG (`rag=True` -> Query Rewrite + Graph DB + Reranker + Enhanced LLM)

In [ ]:
rag_payload = {
    "query": "Hạn mức giao đất nông nghiệp cho cá nhân theo luật đất đai 2024 quy định như thế nào?",
    "rag": True,
    "top_k": 3,
    "session_id": "test_rag_002",
    "metadata": {
        "user_id": "vinh_01"
    }
}

res_rag = client.post("/query", json=rag_payload)
print(f"Status Code: {res_rag.status_code}")
data = res_rag.json()
print(json.dumps(data, indent=2, ensure_ascii=False))